## PointNet++ MSG on ModelNet40 with Colab T4 Training

In [4]:
# ---- Install non-default packages ----
!pip install -q trimesh

# ---- Standard imports ----
import os
import time
import io
import glob
import urllib.request
import zipfile
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import trimesh

# ---- Reproducibility ----
SEED = 1234
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ---- Device check ----
if torch.cuda.is_available():
    device = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU enabled : {gpu_name}  ({gpu_mem:.1f} GB)")
else:
    raise RuntimeError("No GPU detected! Runtime -> Change runtime type -> T4 GPU")

# ---- Paths (Colab working directory is /content) ----
WORK_DIR       = "/content"
DATA_DIR_M40   = os.path.join(WORK_DIR, "ModelNet40")
ZIP_PATH_M40   = os.path.join(WORK_DIR, "ModelNet40.zip")
CACHE_DIR_M40  = os.path.join(WORK_DIR, "cache_modelnet40_1024pts")
CHECKPOINT_DIR = os.path.join(WORK_DIR, "checkpoints")
os.makedirs(CACHE_DIR_M40, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"\nPyTorch version : {torch.__version__}")
print(f"CUDA version    : {torch.version.cuda}")
print(f"Device          : {device}")
print(f"Working dir     : {WORK_DIR}")
print(f"\nReady. Continue to Cell 3 (download ModelNet40).")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 741.0/741.0 kB 15.9 MB/s eta 0:00:00
GPU enabled : Tesla T4  (14.6 GB)

PyTorch version : 2.11.0+cu128
CUDA version    : 12.8
Device          : cuda
Working dir     : /content

Ready. Continue to Cell 3 (download ModelNet40).


## Downloading and extracting ModelNet40

In [5]:
URLS_M40 = [
    "http://modelnet.cs.princeton.edu/ModelNet40.zip",
    "https://3dvision.princeton.edu/projects/2014/3DShapeNets/ModelNet40.zip",
]


def download_with_progress(url, dest):
    """Download with a percentage progress bar that updates in place."""
    def hook(block_num, block_size, total_size):
        downloaded = block_num * block_size
        if total_size > 0:
            percent = min(100, downloaded * 100 / total_size)
            mb_done  = downloaded / 1024**2
            mb_total = total_size / 1024**2
            print(f"\r  {percent:5.1f}%  ({mb_done:7.1f} / {mb_total:7.1f} MB)", end="")
    urllib.request.urlretrieve(url, dest, reporthook=hook)
    print()


# ---- A. Download (skip if zip exists and is at least 1 GB) ----
if os.path.exists(ZIP_PATH_M40) and os.path.getsize(ZIP_PATH_M40) > 1e9:
    size_mb = os.path.getsize(ZIP_PATH_M40) / 1024**2
    print(f"Zip already exists ({size_mb:.1f} MB) -- skipping download")
else:
    print("Downloading ModelNet40.zip (~1.7 GB)...")
    download_success = False
    for url in URLS_M40:
        try:
            print(f"  Trying: {url}")
            download_with_progress(url, ZIP_PATH_M40)
            download_success = True
            print(f"  OK")
            break
        except Exception as e:
            print(f"  FAILED: {e}")
            if os.path.exists(ZIP_PATH_M40):
                os.remove(ZIP_PATH_M40)
    if not download_success:
        raise RuntimeError("All Princeton URLs failed. Try later.")


# ---- B. Extract (skip if a known class folder already exists) ----
if os.path.isdir(os.path.join(DATA_DIR_M40, "airplane")):
    print(f"\nModelNet40 already extracted -- skipping")
else:
    print(f"\nExtracting ModelNet40.zip to {WORK_DIR}/ ...")
    with zipfile.ZipFile(ZIP_PATH_M40, "r") as zf:
        zf.extractall(WORK_DIR)
    print(f"  OK")


# ---- C. Verify class folders ----
CLASSES_M40 = sorted([
    d for d in os.listdir(DATA_DIR_M40)
    if os.path.isdir(os.path.join(DATA_DIR_M40, d))
    and not d.startswith(("_", "."))
])
CLASS_TO_IDX_M40 = {c: i for i, c in enumerate(CLASSES_M40)}
IDX_TO_CLASS_M40 = {i: c for c, i in CLASS_TO_IDX_M40.items()}

print(f"\nFound {len(CLASSES_M40)} class folders")
assert len(CLASSES_M40) == 40, f"Expected 40 classes, found {len(CLASSES_M40)}"
print("ModelNet40 ready.")
print(f"First 5 classes: {CLASSES_M40[:5]}")
print(f"Last  5 classes: {CLASSES_M40[-5:]}")

  Trying: http://modelnet.cs.princeton.edu/ModelNet40.zip
  100.0%  ( 1944.7 /  1944.7 MB)
  OK

Extracting ModelNet40.zip to /content/ ...
  OK

Found 40 class folders
ModelNet40 ready.
First 5 classes: ['airplane', 'bathtub', 'bed', 'bench', 'bookshelf']
Last  5 classes: ['toilet', 'tv_stand', 'vase', 'wardrobe', 'xbox']
